In [1]:
import cv2#open cv
import mediapipe as mp#mediapipe-->deteccion de manos
import screen_brightness_control as sbc#control de brillo
from pycaw.pycaw import AudioUtilities, ISimpleAudioVolume#control de audio
import tkinter as tk#interfaz
from tkinter import BooleanVar
import threading#hilos

# Función para setear el volumen
def set_volume(volume_level):
    sessions = AudioUtilities.GetAllSessions()
    for session in sessions:
        volume = session._ctl.QueryInterface(ISimpleAudioVolume)
        volume.SetMasterVolume(volume_level / 100.0, None)

# Función para iniciar la detección de manos
def start_detection():
    global running
    running = True
    threading.Thread(target=hand_detection).start()

# Función para detener la detección de manos
def stop_detection():
    global running
    running = False

# Función principal de detección de manos
def hand_detection():
    global running, volume_label, brightness_label, show_video_var
    #modelo de mediapipe de la mano
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)
    cap = cv2.VideoCapture(0)

    while running and cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)#imagen captada
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)#paso a rgb, open cv lo trata con bgr
        results = hands.process(rgb_frame)#detecto la mano con metodo de mediapipe

        if results.multi_hand_landmarks:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):#analizo todos los puntos 
                mp_drawing = mp.solutions.drawing_utils#muestro la mano con los resultados
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                label = handedness.classification[0].label#distingo entre derecha e izquierda

                finger_tips_ids = [4, 8, 12, 16, 20]
                finger_count_right = 0
                finger_count_left = 0

                for i, tip_id in enumerate(finger_tips_ids):
                    if label == 'Right':
                        if tip_id == 4:#caso pulgar
                            if hand_landmarks.landmark[2].x < hand_landmarks.landmark[18].x:
                                if hand_landmarks.landmark[tip_id].x < hand_landmarks.landmark[tip_id - 1].x:
                                    finger_count_right += 1
                        else:#resto de dedos
                            if hand_landmarks.landmark[tip_id].y < hand_landmarks.landmark[tip_id - 2].y:
                                finger_count_right += 1
                    else:#mano izquierda
                        if tip_id == 4:
                            if hand_landmarks.landmark[2].x > hand_landmarks.landmark[18].x:
                                if hand_landmarks.landmark[tip_id].x > hand_landmarks.landmark[tip_id - 1].x:
                                    finger_count_left += 1
                        else:
                            if hand_landmarks.landmark[tip_id].y < hand_landmarks.landmark[tip_id - 2].y:
                                finger_count_left += 1

                position = (10, 30) if label == 'Right' else (10, 70)
                text = f'{label}: {finger_count_right} dedos levantados' if label == 'Right' else f'{label}: {finger_count_left} dedos levantados' 
                cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

                bright = 20 * finger_count_left
                vol = 20 * finger_count_right

                if label == 'Right' and results.multi_hand_landmarks:
                    set_volume(vol)
                    volume_label.config(text=f"Volumen: {vol}%")
                if label == 'Left' and results.multi_hand_landmarks:
                    sbc.set_brightness(bright)
                    brightness_label.config(text=f"Brillo: {bright}%")

        if show_video_var.get():
            cv2.imshow('Hand Detection', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    hands.close()

# Configuración de la interfaz gráfica
app = tk.Tk()
app.title("Control de Volumen y Brillo")
app.geometry("400x300")

# Etiquetas y botones de control
volume_label = tk.Label(app, text="Volumen: 50%", font=("Helvetica", 12))
volume_label.pack(pady=10)

brightness_label = tk.Label(app, text="Brillo: 50%", font=("Helvetica", 12))
brightness_label.pack(pady=10)

start_button = tk.Button(app, text="Iniciar Detección", command=start_detection, bg="green", fg="white")
start_button.pack(pady=10)

stop_button = tk.Button(app, text="Detener Detección", command=stop_detection, bg="red", fg="white")
stop_button.pack(pady=10)

# Checkbox para mostrar/ocultar video
show_video_var = BooleanVar(value=True)
show_video_checkbox = tk.Checkbutton(app, text="Mostrar Video", variable=show_video_var)
show_video_checkbox.pack(pady=10)

# Variable para controlar el hilo de detección
running = False

# Ejecuta la interfaz
app.mainloop()


C:\Users\brute\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
